In [1]:
import glob
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch

from dataset import OptionsDataModule
from model import OptionNetModule

warnings.filterwarnings("ignore")

In [2]:
# --- Constants ---
CHECKPOINT_PATH = './checkpoints/run_20260208_013330_114849/epoch=99-step=84700.ckpt'
DATA_DIR = "./data/108105"
SOFR_PATH = "./data/sofr.csv"
BATCH_SIZE = 512
SELECTED_DATE = "2025-08-29"  # Set to None to use latest available date
CP_FLAG = "C"
RISK_FREE_RATE = 0.04  # fallback SOFR when missing

In [3]:
def choose_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def expected_raw_feature_count(model):
    first_linear = model.model.model[0]
    return int(first_linear.in_features) + 1

In [4]:
print("Loading Data Module...")
data_module = OptionsDataModule(DATA_DIR, sofr_path=SOFR_PATH, batch_size=BATCH_SIZE)
data_module.setup("fit")

print(f"Loading Model from {CHECKPOINT_PATH}...")
model = OptionNetModule.load_from_checkpoint(CHECKPOINT_PATH)
model.eval()

device = choose_device()
model = model.to(device)

print("Loading raw data for selected date...")
raw_df = pd.concat(
    [pd.read_csv(path) for path in sorted(glob.glob(f"{DATA_DIR}/*.csv"))],
    ignore_index=True,
)
raw_df["date"] = pd.to_datetime(raw_df["date"])

if "cp_flag" in raw_df.columns:
    raw_df = raw_df[raw_df["cp_flag"] == CP_FLAG].copy()

sofr_df = pd.read_csv(SOFR_PATH)
sofr_df["date"] = pd.to_datetime(sofr_df["date"])
sofr_df["sofr"] = sofr_df["sofr"] / 100.0

raw_df = pd.merge(raw_df, sofr_df[["date", "sofr"]], on="date", how="left")
raw_df["sofr"] = raw_df["sofr"].fillna(RISK_FREE_RATE)
raw_df["T"] = raw_df["T"] / 365.0
raw_df["vix"] = raw_df["vix"] / 100.0

if raw_df.empty:
    raise ValueError(f"No rows found under {DATA_DIR}.")

target_date = pd.to_datetime(SELECTED_DATE) if SELECTED_DATE else raw_df["date"].max()
daily = raw_df[raw_df["date"] == target_date].copy()
if daily.empty:
    available = raw_df["date"].dt.date.drop_duplicates().sort_values().astype(str).tail(10).tolist()
    raise ValueError(
        f"No rows found for SELECTED_DATE={target_date.date()}. "
        f"Last available dates: {available}"
    )

S_fixed = float(daily["S"].median())
vix_fixed = float(daily["vix"].median())
sofr_fixed = float(daily["sofr"].median())

raw_feature_count = expected_raw_feature_count(model)
print(f"Model expects {raw_feature_count} raw features")

hv_cols = ["hv_10", "hv_14", "hv_30", "hv_60", "hv_91"]
h_vol_values = {}
if raw_feature_count in (9, 10):
    missing_hv = [col for col in hv_cols if col not in daily.columns]
    if missing_hv:
        raise ValueError(f"Missing HV columns required by model: {missing_hv}")
    h_vol_values = daily[hv_cols].median().astype(float).to_dict()
    if any(pd.isna(v) for v in h_vol_values.values()):
        raise ValueError(f"Historical vol is missing for date {target_date.date()} in columns {hv_cols}.")

if pd.isna(vix_fixed):
    raise ValueError(f"VIX is missing for date {target_date.date()}.")
if pd.isna(sofr_fixed):
    sofr_fixed = RISK_FREE_RATE

print(f"Using date={target_date.date()}, S={S_fixed:.2f}, VIX={vix_fixed:.4f}, SOFR={sofr_fixed:.4f}")
if h_vol_values:
    print("Historical vol snapshot:", h_vol_values)

# --- Generate Grid ---
print("Generating Grid...")
k_min = 0.6 * S_fixed
k_max = 1.5 * S_fixed
moneyness_min = S_fixed / k_max
moneyness_max = S_fixed / k_min
moneyness_values = np.linspace(moneyness_min, moneyness_max, 30)

T_years = np.linspace(10 / 365.0, 730 / 365.0, 30)

# Create meshgrid
moneyness_grid, T_grid = np.meshgrid(moneyness_values, T_years)
K_grid = S_fixed / moneyness_grid

# Flatten for model input
K_flat = K_grid.flatten()
T_flat = T_grid.flatten()

feature_blocks = [
    torch.full((len(K_flat), 1), S_fixed, dtype=torch.float32),
    torch.tensor(K_flat, dtype=torch.float32).reshape(-1, 1),
    torch.tensor(T_flat, dtype=torch.float32).reshape(-1, 1),
    torch.full((len(K_flat), 1), vix_fixed, dtype=torch.float32),
]

if raw_feature_count in (9, 10):
    hv_tensors = [
        torch.full((len(K_flat), 1), float(h_vol_values[col]), dtype=torch.float32)
        for col in hv_cols
    ]
    feature_blocks.extend(hv_tensors)

if raw_feature_count == 10:
    feature_blocks.append(torch.full((len(K_flat), 1), sofr_fixed, dtype=torch.float32))
elif raw_feature_count not in (4, 9):
    raise ValueError(f"Unsupported raw feature count {raw_feature_count}. Expected one of 4, 9, 10.")

x_input = torch.cat(feature_blocks, dim=1).to(device)

print("Predicting Prices...")
preds, greeks = model(x_input)

predicted_prices = (preds.detach().cpu().numpy().flatten() * K_flat)

delta_grid = greeks["delta"].detach().cpu().numpy().reshape(K_grid.shape)
gamma_grid = greeks["gamma"].detach().cpu().numpy().reshape(K_grid.shape)
theta_grid = greeks["theta"].detach().cpu().numpy().reshape(K_grid.shape)

price_grid = np.array(predicted_prices).reshape(K_grid.shape)


def plot_surface(z_grid, title, zaxis_title, colorbar_title):
    fig = go.Figure(data=[
        go.Surface(
            x=moneyness_grid,
            y=T_grid,
            z=z_grid,
            colorscale='Viridis',
            colorbar_title=colorbar_title,
            opacity=0.9,
            hovertemplate=(
                "Moneyness (S/K): %{x:.4f}<br>" +
                "Time (Years): %{y:.2f}<br>" +
                f"{zaxis_title}: %{{z:.6f}}<extra></extra>"
            )
        ),
    ])

    fig.update_layout(
        title={
            'text': title,
            'y': 0.9,
            'x': 0.5,
            'xanchor': 'center',
            'yanchor': 'top'
        },
        scene=dict(
            xaxis_title='Moneyness (S/K)',
            yaxis_title='Time to Maturity (Years)',
            zaxis_title=zaxis_title,
            aspectratio=dict(x=1, y=1, z=0.6),
            camera=dict(
                eye=dict(x=1.5, y=1.5, z=0.5)
            )
        ),
        width=1000,
        height=700,
        margin=dict(l=50, r=50, b=0, t=50)
    )
    fig.show()


meta = f"date={target_date.date()}, S={S_fixed:.2f}, VIX={vix_fixed:.4f}, SOFR={sofr_fixed:.4f}"

print("Plotting Surface...")
plot_surface(
    price_grid,
    f"Price Surface ({meta})",
    'Price',
    'Predicted Price'
)

print("Plotting Delta Surface...")
plot_surface(
    delta_grid,
    f"Delta Surface ({meta})",
    'Delta',
    'Delta'
)

print("Plotting Gamma Surface...")
plot_surface(
    gamma_grid,
    f"Gamma Surface ({meta})",
    'Gamma',
    'Gamma'
)

print("Plotting Theta Surface...")
plot_surface(
    theta_grid,
    f"Theta Surface ({meta})",
    'Theta',
    'Theta'
)

Loading Data Module...
Date split -> Train: 352 | Validation: 20 | Test: 43
Rows -> Train: 866851 | Validation: 47386 | Test: 103172
Loading Model from ./checkpoints/run_20260208_013330_114849/epoch=99-step=84700.ckpt...
Loading raw data for selected date...
Model expects 10 raw features
Using date=2025-08-29, S=6460.26, VIX=0.1536, SOFR=0.0444
Historical vol snapshot: {'hv_10': 0.109674, 'hv_14': 0.102552, 'hv_30': 0.117107, 'hv_60': 0.093979, 'hv_91': 0.095489}
Generating Grid...
Predicting Prices...
Plotting Surface...


Plotting Delta Surface...


Plotting Gamma Surface...


Plotting Theta Surface...


In [5]:
print("Collecting validation/testing price errors...")
torch.set_grad_enabled(True)


def collect_price_errors(loader, split_name):
    real_prices = []
    model_prices = []

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        preds_norm, _ = model(xb)
        k = xb[:, 1:2]

        real_batch = (yb * k).detach().cpu().numpy().ravel()
        model_batch = (preds_norm * k).detach().cpu().numpy().ravel()

        real_prices.append(real_batch)
        model_prices.append(model_batch)

    if not real_prices:
        raise ValueError(f"No rows available for {split_name} set.")

    real_prices = np.concatenate(real_prices)
    model_prices = np.concatenate(model_prices)
    errors = model_prices - real_prices

    mae = np.mean(np.abs(errors))
    rmse = np.sqrt(np.mean(errors ** 2))
    print(f"{split_name}: n={len(errors):,}, mean={errors.mean():.4f}, std={errors.std():.4f}, MAE={mae:.4f}, RMSE={rmse:.4f}")
    return errors


val_errors = collect_price_errors(data_module.val_dataloader(), "Validation")
test_errors = collect_price_errors(data_module.test_dataloader(), "Testing")

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=val_errors,
    name="Validation Error",
    opacity=0.65,
    histnorm="percent",
    nbinsx=80,
    marker_color="#1f77b4",
))
fig.add_trace(go.Histogram(
    x=test_errors,
    name="Testing Error",
    opacity=0.65,
    histnorm="percent",
    nbinsx=80,
    marker_color="#ff7f0e",
))

fig.add_vline(x=0.0, line_dash="dash", line_color="black")
fig.update_layout(
    title="Error Distribution: Model Price - Real Price (Validation vs Testing)",
    xaxis_title="Pricing Error",
    yaxis_title="Percent of Samples (%)",
    barmode="overlay",
    template="plotly_white",
    width=1000,
    height=500,
)
fig.show()

Validation: n=47,386, mean=0.0454, std=12.8071, MAE=8.1364, RMSE=12.8072
Testing: n=103,172, mean=0.6001, std=13.3664, MAE=9.4989, RMSE=13.3799
